# Subject 5 · The character network of *Les Misérables*: centrality and communities

**Course "Artificial Intelligence"** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Mini-project, subject 5** (social network analysis) · **Duration:** about 1 h 30

**What you will do**
- load the network of the characters of Victor Hugo's *Les Misérables* (1862), which comes with the NetworkX library, and explore it character by character;
- measure the position of each one: degree centrality (who is the most connected?) and betweenness centrality (who acts as a bridge?), then draw the network;
- find communities of characters and interpret them in literary terms: plots, episodes, the structural role of characters.

**How to use this notebook**
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. You never have to write code. Change only the fields of the cells titled "TO CHANGE". In Colab, the menu "Runtime → Run all" runs the whole notebook at once.

Before each code cell, a question asks you to **predict** the result: write down your answer, run the cell, then compare with the section "What you should see".

> **The data.** This network is not made up: Donald Knuth built it for his book *The Stanford GraphBase* (1993) by noting, chapter by chapter, the characters present. Two characters are connected if they appear in the same chapter; the **weight** of the link is the number of chapters they share. The novel is in the public domain; the names are written without accents (`Thenardier` for Thénardier).

## Step 1 · Load the tools

This cell loads NetworkX (graphs), pandas (tables) and matplotlib (drawings).

**Before you run it:** under what short name will we use NetworkX in the rest of the notebook?

In [ ]:
#@title Load the tools (libraries)
import networkx as nx                 # graphs: nodes and links
import pandas as pd                   # data tables
import matplotlib.pyplot as plt       # drawings

# Colors of the communities, always in the same order
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
print("Tools loaded. NetworkX version:", nx.__version__)

**What you should see:** `Tools loaded. NetworkX version:` followed by a version number (3.x; the exact number depends on your installation). We will write `nx` for NetworkX.

**Question:** a graph is made of "nodes" and "links". In a novel, what can the nodes be? What can the links be? Give two possibilities for each.

## Step 2 · Load the network of *Les Misérables*

This cell loads the network and displays its size: the number of characters (nodes), the number of links and the **density**, that is, the share of possible links that really exist (1 = everyone is connected to everyone).

**Before you run it:** how many characters do you think Knuth kept: about thirty, nearly 80, or more than 200? Will the density be close to 0 or to 1?

In [ ]:
#@title Load the network and display its size
G = nx.les_miserables_graph()          # network provided with NetworkX
print("Characters (nodes):", G.number_of_nodes())
print("Links:             ", G.number_of_edges())
print("Density:           ", round(nx.density(G), 3))
print("Some characters:", ", ".join(sorted(G.nodes)[:12]), "…")

**What you should see:** 77 characters, 254 links and a density of 0.087: fewer than 9% of the possible pairs of characters meet in a chapter. The first names, in alphabetical order: Anzelma, Babet, Bahorel, Bamatabois, BaronessT, Blacheville, Bossuet, Boulatruelle, Brevet, Brujon, Champmathieu, Champtercier.

**Question:** is a network of characters with a low density the sign of a "badly built" novel? Think of the length of *Les Misérables* and of the number of places and periods it covers.

## Step 3 · Explore a character

Choose a character from the list (or type another name exactly as in the data). The cell displays the number of characters they are connected to, then their ten most frequent neighbors, with the number of shared chapters.

**Before you run it:** with whom does Jean Valjean share the most chapters? Javert, Cosette or Marius?

In [ ]:
#@title TO CHANGE: choose a character, then run (their neighbors)
character = "Valjean" #@param ["Valjean", "Javert", "Cosette", "Marius", "Fantine", "Gavroche", "Myriel", "Thenardier", "Eponine", "Enjolras"] {allow-input: true}
neighbors = G[character]               # the characters connected to this one
print(character, "is connected to", len(neighbors), "characters.")
table = pd.DataFrame([(v, neighbors[v]["weight"]) for v in neighbors],
                     columns=["neighbor", "shared chapters"])
print(table.sort_values("shared chapters", ascending=False).head(10).to_string(index=False))

**What you should see:** `Valjean is connected to 36 characters.` His most frequent neighbors: Cosette (31 chapters), Marius (19), Javert (17), Thenardier (12), Fantine (9), Fauchelevent (8), MmeThenardier (7), Myriel (5), Enjolras (4), MmeMagloire (3). It is Cosette, his adopted daughter, and not Javert, his pursuer, who most often goes with Valjean.

**Question:** choose Javert, then Gavroche. Which one has more neighbors? Do Gavroche's neighbors belong to the same "world" as Javert's?

## Step 4 · Degree centrality: who is the most connected?

The **degree** of a character is the number of characters they are connected to. **Degree centrality** divides it by the number of other characters (76): it is the share of the network that the character meets directly.

**Before you run it:** who will come second after Valjean? A main character (Marius, Cosette, Javert) or a character you meet everywhere?

In [ ]:
#@title Compute the degree and degree centrality of each character
links = dict(G.degree())                        # number of connected characters
centralities = pd.DataFrame({"links": links, "degree": nx.degree_centrality(G)}).round(3)
top = centralities.sort_values("links", ascending=False).head(10)
print(top)
to_draw = top.sort_values("links")              # from smallest to largest, for the drawing
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(to_draw.index, to_draw["links"], color=COLORS[0])
ax.set_xlabel("Number of connected characters (degree)")
ax.set_title("The 10 most connected characters")
plt.show()

**What you should see:** Valjean (36 links, centrality 0.474: he meets nearly half of the characters), then Gavroche (22), Marius (19), Javert (17), Thenardier (16), Enjolras and Fantine (15), Bossuet and Courfeyrac (13), Joly (12). Cosette is not in the top ten.

**Question:** Gavroche comes before Marius, Javert and Cosette. What in his role as a Paris street urchin (a *gamin de Paris*) explains that he meets so many characters?

## Step 5 · Betweenness centrality: who acts as a bridge?

To go from one character to another in the network, we follow the shortest path of links. **Betweenness centrality** measures the share of these shortest paths that go through a character: a character with high betweenness connects groups that, without them, would not touch.

**Before you run it:** can a character who appears only in the first book of the novel have a high betweenness?

In [ ]:
#@title Compute betweenness centrality
centralities["betweenness"] = pd.Series(nx.betweenness_centrality(G)).round(3)
centralities.sort_values("betweenness", ascending=False).head(10)

**What you should see:** Valjean overwhelms everyone (0.570: more than half of the shortest paths go through him). Then come **Myriel** (0.177), Gavroche (0.165), Marius (0.132), Fantine (0.130), Thenardier (0.075), Javert (0.054), MlleGillenormand (0.048), Enjolras (0.043) and Tholomyes (0.041). Myriel, the bishop of Digne, has only 10 links but he is the second "bridge" of the novel.

**Question:** Javert has 17 links but a low betweenness (0.054). What can we conclude about his place in the structure of the novel, compared with Myriel's?

This chart places each character according to their two centralities: degree on the horizontal axis, betweenness on the vertical axis. The names of the eight biggest "bridges" are written.

**Before you run it:** where will Myriel be on the chart: top right, top left, or at the bottom?

In [ ]:
#@title Compare the two centralities on a chart
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(centralities["degree"], centralities["betweenness"], s=40, color=COLORS[0], alpha=0.7)
for name in centralities.sort_values("betweenness").tail(8).index:     # the 8 biggest "bridges"
    ax.annotate(name, (centralities.loc[name, "degree"], centralities.loc[name, "betweenness"]),
                xytext=(5, 3), textcoords="offset points")
ax.set_xlabel("Degree centrality (share of characters met)")
ax.set_ylabel("Betweenness centrality (bridge role)")
ax.set_title("Very connected, or a bridge?")
plt.show()

**What you should see:** a cloud of 77 points packed at the bottom left (most characters have few links and are a bridge for nobody), Valjean alone at the very top right, and Myriel **at the top left**: few links, many paths going through him. Gavroche, Marius and Fantine form an intermediate group.

**Question:** in literary terms, what does Myriel's position mean? Think of the first book of the novel ("A Just Man") and of the scene of the candlesticks.

## Step 6 · Draw the network

NetworkX places the characters with a "spring" layout: links pull connected characters towards each other, and characters push each other away. The size of a point depends on its degree, the thickness of a line on the number of shared chapters. Only characters with at least 10 links are named. `seed=3` makes the drawing reproducible.

**Before you run it:** will Valjean be in the center of the drawing or on an edge? Will we see separate groups?

In [ ]:
#@title Draw the network (size = number of links, thickness = shared chapters)
position = nx.spring_layout(G, seed=3)                  # "spring" layout
sizes = [25 * links[p] for p in G.nodes]
widths = [0.3 + G[u][v]["weight"] / 6 for u, v in G.edges]
main_names = {p: p for p in G.nodes if links[p] >= 10}  # names displayed
fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_edges(G, position, width=widths, alpha=0.4, edge_color="gray", ax=ax)
nx.draw_networkx_nodes(G, position, node_size=sizes, node_color=COLORS[0], ax=ax)
nx.draw_networkx_labels(G, position, labels=main_names, font_size=9, ax=ax)
ax.set_title("Les Misérables: who appears with whom? (Knuth's data, 1993)")
ax.axis("off")
plt.show()

**What you should see:** SEE_DRAWING

**Question:** the thickest line connects Valjean and Cosette (31 chapters). Which other thick line can you spot, and which relationship in the novel does it represent?

## Step 7 · Communities

A **community** is a group of characters that are closely connected to each other and loosely connected to the rest. The function `greedy_modularity_communities` looks for them by merging groups step by step, as long as the **modularity** (a score that measures whether links tend to stay inside the groups) goes up. You can choose to take the weights into account (the number of shared chapters) or to count all links in the same way.

**Before you run it:** how many communities do you expect? Will Valjean and Javert be in the same one?

In [ ]:
#@title TO CHANGE: take the weights into account or not, then run (communities)
with_weights = True #@param {type:"boolean"}
weight = "weight" if with_weights else None  # None: all links count the same
communities = nx.community.greedy_modularity_communities(G, weight=weight)
print(len(communities), "communities, modularity =",
      round(nx.community.modularity(G, communities, weight=weight), 3))
for number, members in enumerate(communities):
    print(f"\nCommunity {number} ({len(members)} characters):")
    print("   ", ", ".join(sorted(members)))

**What you should see:** SEE_COMMUNITIES

**Question:** which community is Marius in? And Enjolras? What does this separation say about Marius being torn between his revolutionary friends and his love for Cosette?

This cell draws the network again with the **same layout** as in step 6, but each character takes the color of their community.

**Before you run it:** will the communities form clearly separate areas in the drawing, or will they be mixed?

In [ ]:
#@title Draw the network colored by community
color_of = {}
for number, members in enumerate(communities):
    for p in members:
        color_of[p] = COLORS[number]
fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_edges(G, position, width=widths, alpha=0.3, edge_color="gray", ax=ax)
nx.draw_networkx_nodes(G, position, node_size=sizes, node_color=[color_of[p] for p in G.nodes], ax=ax)
nx.draw_networkx_labels(G, position, labels=main_names, font_size=9, ax=ax)
for number in range(len(communities)):                  # legend: one color per community
    ax.scatter([], [], color=COLORS[number], label=f"Community {number}")
ax.legend(loc="lower left")
ax.set_title(f"{len(communities)} communities in the network of Les Misérables")
ax.axis("off")
plt.show()

**What you should see:** SEE_COMMUNITY_DRAWING

**Question:** find in the drawing a community that touches the rest of the network through only one or two characters. Which ones?

### Reading the communities in literary terms

LITERARY_READING

## Step 8 · Remove a character: Moretti's experiment

In "Network Theory, Plot Analysis" (2011), Franco Moretti represented *Hamlet* as a network, then observed what happens to the play when a character is removed. Let us do the same: the cell removes the chosen character and counts the **pieces** of the network that are no longer connected to each other.

**Before you run it:** without Valjean, will the network stay in one piece? If not, which characters will end up isolated?

In [ ]:
#@title TO CHANGE: remove a character, then run
remove = "Valjean" #@param ["Valjean", "Javert", "Cosette", "Marius", "Fantine", "Gavroche", "Myriel", "Thenardier", "Eponine", "Enjolras"] {allow-input: true}
H = G.copy()                           # a copy: the network G stays intact
H.remove_node(remove)
pieces = sorted(nx.connected_components(H), key=len, reverse=True)
print(f"Without {remove}, the network splits into {len(pieces)} piece(s).")
print("The largest piece keeps", len(pieces[0]), "characters out of", H.number_of_nodes())
for piece in pieces[1:]:
    print("   cut off:", ", ".join(sorted(piece)))

**What you should see:** without Valjean, the network splits into **7 pieces**. The largest keeps 61 characters out of 76. Everyone from Digne is cut off (Myriel, MlleBaptistine, MmeMagloire, Napoleon, Champtercier, Cravatte, Count, CountessDeLo, Geborand, OldMan), as well as five characters whom only Valjean meets: Gervais (the little Savoyard whose coin he steals), Isabeau, Labarre (the innkeeper in Digne who throws him out), MmeDeR and Scaufflaire (who hires out to him the carriage for Arras).

**Question:** now remove Myriel, then Gavroche, then Marius. Which of these removals cuts **nothing** off? What does this say about the difference between a character who matters for the plot and a character who is essential to the structure?

## Your turn

Change only the fields, then run the cells again from the one you changed (in Colab: "Runtime → Run after").
1. **Step 3:** compare the neighbors of Fantine and Cosette, mother and daughter. Do they share neighbors? What does this say about the way the novel is built in successive periods?
2. **Step 7:** untick `with_weights` and run step 7 and the following cells again. How many communities do you get? Which community disappears, and where do its members go?
3. **Step 8:** type `Fantine`, then `Thenardier` in the field (or choose them from the list). Does removing them split the network? Compare with their betweenness (step 5).

## For the mini-project: your own network and Gephi

Subject 5 suggests, for example, studying the network of the characters in *Nedjma* (Kateb Yacine, 1956). The novel is protected by copyright: share your link tables, never the digitized text. To build your network:
- first write down your **rule**: what is a character, what is a link (appearing in the same section, talking to each other, writing to each other)?
- in a spreadsheet, make a link table with **three columns** named `source`, `cible` and `poids` (as on the course site: source, target, weight; one row per pair of characters, the weight = number of shared sections), then save it as CSV;
- tick the box below and run the cell: in Colab, a window asks you for the file; in Jupyter, put it next to the notebook under the name `my_links.csv`;
- then run steps 3 to 8 again, typing the name of one of your characters in the fields of steps 3 and 8.

**Before you run it:** if you leave the box unticked, what will the cell do?

In [ ]:
#@title TO CHANGE: load your own link table (CSV file)
load_my_file = False #@param {type:"boolean"}
if load_my_file:
    try:
        from google.colab import files       # only exists in Colab
        upload = files.upload()              # window to choose the file
        file_name = list(upload.keys())[0]
    except ImportError:                      # Jupyter: file placed next to the notebook
        file_name = "my_links.csv"
    link_table = pd.read_csv(file_name).rename(columns={"poids": "weight"})
    G = nx.from_pandas_edgelist(link_table, "source", "cible", edge_attr="weight")
    print("Your network:", G.number_of_nodes(), "characters,", G.number_of_edges(), "links")
else:
    print("Box not ticked: the notebook keeps the network of Les Misérables.")

**What you should see:** with the box unticked, `Box not ticked: the notebook keeps the network of Les Misérables.` With your file, the number of characters and links in your network.

The mini-project also asks for a visualization with **Gephi**, free software that lays out networks. The next cell saves the network (with the community number of each character) in the GEXF format, which Gephi can open.

**Before you run it:** where will the file be saved in Colab? And in Jupyter?

In [ ]:
#@title TO CHANGE: export the network for Gephi (GEXF file)
export = False #@param {type:"boolean"}
if export:
    for number, members in enumerate(communities):
        for p in members:
            G.nodes[p]["community"] = number         # Gephi can color by community
    nx.write_gexf(G, "network.gexf")
    try:
        from google.colab import files
        files.download("network.gexf")             # Colab: download to your computer
    except ImportError:
        print("File network.gexf saved next to the notebook.")
else:
    print("Box not ticked: no file is created.")

**What you should see:** with the box unticked, `Box not ticked: no file is created.` Ticked, the file `network.gexf` is downloaded (Colab) or saved next to the notebook (Jupyter). In Gephi: "File → Open", then color the nodes by the `community` attribute.

**Question:** NetworkX computes, Gephi lays out. What advantage do you see in keeping the calculations in a notebook that others can run again?

## To hand in

Answer in writing, in English, in a few lines for each question.
1. **Centrality.** Compare Valjean, Myriel, Javert and Gavroche using degree and betweenness. For each one, say in one sentence what these figures reveal about their role in the structure of the novel.
2. **Communities.** Choose two communities found in step 7. Link each one to an episode, a place or a plot line of *Les Misérables*, and point out a character whose place surprises you.
3. **Limitations.** Knuth connects two characters present in the same chapter. Give two limitations of this rule (for example: being in the same chapter without meeting, chapters of very different lengths, digressions without characters such as Waterloo) and suggest another link rule for your own corpus.

## Key points

- A **graph** represents relationships: **nodes** (characters) connected by **links** (here, appearing in the same chapter), with a **weight**. Everything depends on the rule that defines a link.
- **Degree centrality** measures how many characters one meets; **betweenness centrality** measures the **bridge** role between groups. A character with few links can be an essential bridge (Myriel).
- **Communities** reveal groups of characters that often match episodes or plot lines; figures do not replace reading: they must be checked against the text.

**Lesson on the course site:** [Mini-project, subject 5: social network analysis](https://progremer04.github.io/canva-au-for-ang-/en/#mp-sujet-5)